In [6]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

BUSINESS_METRICS = "business_metrics_gold"
GOLD = "city_metrics_gold"

df = spark.table(BUSINESS_METRICS)

df_city = (
    df.groupBy("state", "city")
    .agg(
        F.count("business_id").alias("business_count"),
        F.sum("review_count_total").alias("review_count_total"),
        F.round(
            F.sum(F.col("avg_rating") * F.col("review_count_total")) / F.sum(F.col("review_count_total")),
            2
        ).alias("weighted_avg_rating")
    )
)

if spark.catalog.tableExists(GOLD):
    delta = DeltaTable.forName(spark, GOLD)

    (
        delta.alias("t")
        .merge(
            df_city.alias("c"),
            "t.state = c.state and t.city = c.city"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        df_city.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD)
    )


StatementMeta(, 797356da-56b4-433f-b098-f39bef52d5b0, 8, Finished, Available, Finished, False)